# 装箱问题(BPP)

**类别：** 装箱

来源：[https://www.hexaly.com/templates/bin-packing-problem-bpp](https://www.hexaly.com/templates/bin-packing-problem-bpp)


## 问题描述

**在装箱问题(Bin Packing Problem, BPP)**中,若干已知重量的物品必须被分配到具有相同容量的箱子中。每个物品必须恰好放入一个箱子中,且每个箱子内物品的总重量不得超过其容量。目标是最小化所使用的箱子数量。

	

### 学习要点

- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 以建模每个箱子内的物品
- 定义一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个箱子的总重量


## 数据

所提供的装箱问题(BPP)算例来自 [BPPLIB](http://or.dei.unibo.it/library/bpplib) 中的 Falkenauer 算例。数据文件的格式如下:

- 第一行:物品数量
- 第二行:箱子容量
- 接下来每一行对应一个物品的重量


## 建模方法

装箱问题(BPP)的 OptAgent 模型使用 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。对每个箱子,我们定义一个集合变量,表示分配到该箱子中的物品集合。我们对集合变量施加划分约束,以确保每个物品恰好被放入一个箱子中。

我们使用集合上的可变参数 **sum** 算子,以及一个返回任意物品索引对应重量的 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html),来计算每个箱子的总重量。请注意,该 sum 中的项数在搜索过程中会变化,因为集合的大小可以变化。

当一个箱子至少包含一个物品时,它才被实际使用。借助 **count** 算子(返回集合中的元素数量),我们可以检查每个箱子是否被实际使用,从而计算所使用的箱子总数。

该模型对最优箱子数量计算了简单的下界与上界。它仅定义 nbMaxBins 个集合变量,并使用 [hxObjectiveThreshold](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#hxObjectiveThreshold) 在找到使用不超过该下界的箱子的解时停止搜索。


## 结果

> 在 BPPLIB 研究基准上(最多包含 **5,500 个** 物品的算例),Hexaly Optimizer 在 **1 分钟** 运行时间内,于装箱问题(BPP)上达到了 **平均 0.2% 的差距**。[他们的装箱问题(BPP)基准测试页面](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-bin-packing-problem)给出了详细结果。[查看该基准](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-bin-packing-problem)


## Python 实现


In [1]:
import math
from pathlib import Path

from optagent import ModelBuilder, solve


def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]


def main(input_file, output_file=None, time_limit=50):
    file_it = iter(read_integers(input_file))
    nb_items = int(next(file_it))
    bin_capacity = int(next(file_it))
    weights_data = [int(next(file_it)) for _ in range(nb_items)]

    nb_min_bins = int(math.ceil(sum(weights_data) / float(bin_capacity)))
    nb_max_bins = min(nb_items, 2 * nb_min_bins)

    #
    # Declare the optimization model
    #
    model = ModelBuilder()

    # Set decisions: bins[k] represents the items in bin k
    bins = [model.set(nb_items, name=f"bin_{k}") for k in range(nb_max_bins)]

    # Each item must be in one bin and one bin only
    model.constraint(model.partition(bins), name="unique_bin_assignment")

    # Create an array and a function to retrieve the item's weight
    weights = model.array(weights_data)
    weight_lambda = model.lambda_function(lambda i: model.at(weights, i))

    # Weight constraint for each bin
    bin_weights = [model.sum(b, weight_lambda) for b in bins]
    for w in bin_weights:
        model.constraint(w <= bin_capacity, name="bin_weight_capacity")

    # Bin k is used if at least one item is in it
    bins_used = [model.count(b) > 0 for b in bins]

    # Count the used bins
    total_bins_used = model.sum(*bins_used)

    # Minimize the number of used bins
    model.minimize(total_bins_used, name="total_bins_used")

    # Stop once the first objective reaches the trivial lower bound
    solution = solve(
        model,
        time_limit_s=float(time_limit),
        objective_threshold={0: nb_min_bins},
    )
    result_values = solution.values(
        {
            "total_bins_used": total_bins_used,
            **{f"bin_{k}": bin_var for k, bin_var in enumerate(bins)},
        }
    )

    lines = []
    for k in range(nb_max_bins):
        items = sorted(int(item) for item in result_values[f"bin_{k}"])
        if not items:
            continue
        weight_value = int(sum(weights_data[i] for i in items))
        line = f"Bin weight: {weight_value} | Items: " + " ".join(str(i) for i in items)
        lines.append(line)

    header = (
        f"Nb items = {nb_items}; Bin capacity = {bin_capacity}; "
        f"Min bins = {nb_min_bins}; Total bins used = {int(result_values['total_bins_used'])}; "
        f"Status = {solution.status.value}"
    )
    print(header)
    for line in lines:
        print(line)

    if output_file is not None:
        with Path(output_file).open("w", encoding="utf-8") as f:
            f.write(header + "\n")
            f.write("\n".join(lines) + "\n")
    return solution


## 运行实例

Notebook 直接调用 `main` 并显式传入实例路径。以下代码格相互独立，可以按需要单独运行；调整 `time_limit` 可以控制每个实例的求解时间。

In [2]:
from pathlib import Path

INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)


Instances: /Users/dongbox/work/optagent/examples/examples/hexaly/bin_packing_problem_bpp/instances


In [4]:
solution_t120_00 = main(INSTANCE_DIR / "t120_00.txt", time_limit=1)


Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0 objective_threshold={0:40}
Solve summary:
  status: FEASIBLE
  objective: 56
  improvements: initial=2 search=1
  evaluated: 944
  wall_time: 10s
  termination: wall_time_exhausted


Nb items = 120; Bin capacity = 1000; Min bins = 40; Total bins used = 56; Status = feasible
Bin weight: 883 | Items: 41 81 110
Bin weight: 721 | Items: 34 45
Bin weight: 801 | Items: 71 99 113
Bin weight: 779 | Items: 1 75
Bin weight: 696 | Items: 14 116
Bin weight: 998 | Items: 18 57 109
Bin weight: 996 | Items: 6 85 117
Bin weight: 856 | Items: 4 29
Bin weight: 362 | Items: 39
Bin weight: 907 | Items: 38 76 96
Bin weight: 913 | Items: 42 68 97
Bin weight: 799 | Items: 82 83 106
Bin weight: 529 | Items: 77 111
Bin weight: 810 | Items: 73 86 114
Bin weight: 523 | Items: 93 98
Bin weight: 805 | Items: 16 33
Bin weight: 764 | Items: 0 91
Bin weight: 769 | Items: 9 64
Bin weight: 720 | Items: 26 56
Bin weight: 890 | Items: 2 28
Bin weight: 885 | Items: 58 72 74
Bin weight: 748 | Items: 22 52
Bin weight: 790 | Items: 19 31
Bin weight: 980 | Items: 37 50 90
Bin weight: 742 | Items: 11 70
Bin weight: 752 | Items: 13 60
Bin weight: 743 | Items: 3 102
Bin weight: 635 | Items: 53 62
Bin weight:

In [ ]:
solution_t120_05 = main(INSTANCE_DIR / "t120_05.txt", time_limit=1)


In [ ]:
solution_u120_00 = main(INSTANCE_DIR / "u120_00.txt", time_limit=1)
